# Exercise 3 — MCPClient

The **MCPClient** is the consumer side of MCP.  In real MCP it opens a transport to a server subprocess.  Here it holds a reference to an MCPServer (same interface, no subprocess).  For gate testing, inject a `tool_call_fn` instead of a server — the client uses whichever is provided.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class MCPToolDef:
    name: str
    description: str
    input_schema: dict = field(default_factory=dict)

def tool_schema_text(tools):
    lines = []
    for t in tools:
        params = ", ".join(t.input_schema.keys())
        lines.append("- " + t.name + "(" + params + "): " + t.description)
    return "\n".join(lines)
class MCPServer:
    def __init__(self, name="mcp_server"):
        self.name = name
        self._tools = {}
    def tool(self, name, description, schema=None):
        def _decorator(fn):
            self._tools[name] = {"def": MCPToolDef(name, description, schema or {}), "fn": fn}
            return fn
        return _decorator
    def list_tools(self):
        return [e["def"] for e in self._tools.values()]
    def call_tool(self, name, args):
        e = self._tools.get(name)
        if e is None:
            return "Error: unknown tool " + repr(name)
        try:
            return str(e["fn"](**args))
        except Exception as exc:
            return "Error: " + str(exc)

# ── Exercise: implement MCPClient ────────────────────────────────────────────

class MCPClient:
    """Connects to an MCPServer and calls tools on behalf of an agent."""

    def __init__(self, server=None, tool_call_fn=None):
        # TODO: store server and tool_call_fn as instance attributes
        pass

    def list_tools(self):
        # TODO: if self._server is not None, return self._server.list_tools()
        # else return []
        return []

    def call_tool(self, name, args):
        # TODO: if self._tool_call_fn is not None, call it with (name, args)
        # elif self._server is not None, call self._server.call_tool(name, args)
        # else return "Error: no server or tool_call_fn configured"
        return "Error: not implemented"


### Checks

In [ ]:
checks = 0

# helper server for tests
_srv = MCPServer("test")
@_srv.tool("add", "Add two numbers.", {"a": "first", "b": "second"})
def _add(a, b): return str(int(a) + int(b))

# 1 — MCPClient constructs with server=
try:
    client = MCPClient(server=_srv)
    checks += 1; print("✅ 1 MCPClient constructs with server=")
except Exception as e:
    print("❌ 1:", e)

# 2 — list_tools delegates to server
try:
    client = MCPClient(server=_srv)
    tools = client.list_tools()
    assert any(t.name == "add" for t in tools)
    checks += 1; print("✅ 2 list_tools returns server's tool list")
except Exception as e:
    print("❌ 2:", e)

# 3 — call_tool delegates to server
try:
    client = MCPClient(server=_srv)
    assert client.call_tool("add", {"a": "3", "b": "4"}) == "7"
    checks += 1; print("✅ 3 call_tool delegates to server")
except Exception as e:
    print("❌ 3:", e)

# 4 — tool_call_fn overrides server
try:
    mock_fn = lambda name, args: "mocked:" + name
    client = MCPClient(tool_call_fn=mock_fn)
    assert client.call_tool("any_tool", {}) == "mocked:any_tool"
    checks += 1; print("✅ 4 tool_call_fn is used when provided")
except Exception as e:
    print("❌ 4:", e)

# 5 — empty client is safe
try:
    client = MCPClient()
    assert client.list_tools() == []
    result = client.call_tool("x", {})
    assert isinstance(result, str) and len(result) > 0
    checks += 1; print("✅ 5 MCPClient() with no args is safe")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
